# Ежемесячный обзор рынка: макро, денежный рынок, облигации

Notebook собирает данные ЦБ, Мосбиржи и ROISfix, считает изменения и записывает результат в `output/отчёт.xlsx`, на который ссылается презентация.

**Как запускать каждый месяц:**
1. В разделе 0 поменять `REPORT_MONTH`.
2. Закрыть `отчёт.xlsx`, если он открыт в Excel.
3. Меню **Kernel → Restart & Run All** (перезапустить и выполнить всё). Только так: ячейки, запущенные не по порядку, могут оставить в памяти старые данные.

**Структура:**
0. Параметры
1. Подготовка
2. Загрузка данных: ЦБ, Мосбиржа, ROISfix, ручные прогнозы
3. Проверки качества
4. Расчёты: денежный рынок, реальная ставка, КБД, кредитные спреды, прогноз ставки, сводка, ряды для графиков
5. Запись в Excel
6. ИИ-комментарии GigaChat через MCP *(учебная часть)*

## 0. Параметры

Единственная ячейка, которую нужно менять от месяца к месяцу.

In [ ]:
from pathlib import Path

REPORT_MONTH = "2026-09"          # отчётный месяц, ГГГГ-ММ

# Окна истории для графиков
DAILY_WINDOW_MONTHS = 12          # дневные ряды: ставки, курс, ликвидность
MONTHLY_WINDOW_MONTHS = 36        # месячные ряды: инфляция, ключевая ставка

# Папки и файлы
DATA_DIR = Path("data")                    # архив сырых данных, по CSV на показатель
OUTPUT_DIR = Path("output")
REPORT_FILE = OUTPUT_DIR / "отчёт.xlsx"    # на этот файл ссылается презентация
FORECASTS_FILE = Path("прогнозы_ручные.xlsx")   # консенсус и прогноз ЦБ по ставке, заполняется вручную

# ИИ-комментарии (учебная часть, раздел 6): True — запускать, False — пропустить
RUN_AI_COMMENTARY = True

## 1. Подготовка
### 1.1. Библиотеки

In [ ]:
import time
import shutil
from io import StringIO
import xml.etree.ElementTree as ET

import requests
import numpy as np
import pandas as pd
from openpyxl.styles import Font
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)     # показывать текст в таблицах целиком, без обрезки

### 1.2. Ключевые даты

- **отчётная дата** — последний день отчётного месяца;
- **месяц назад** — последний день предыдущего месяца;
- **начало года** — 31 декабря прошлого года;
- **начало истории** — с какой даты нужны данные: самое длинное окно плюс месяц запаса.

Даты календарные. Если на дату не было торгов, позже возьмём последнее доступное значение до неё, поэтому производственный календарь не нужен.

In [ ]:
report_period = pd.Period(REPORT_MONTH, freq="M")

report_date   = report_period.end_time.normalize()
month_ago     = (report_period - 1).end_time.normalize()
year_start    = pd.Timestamp(report_date.year - 1, 12, 31)
history_start = (report_period - MONTHLY_WINDOW_MONTHS - 1).start_time

pd.DataFrame({
    "Дата": [report_date, month_ago, year_start, history_start],
}, index=["Отчётная дата", "Месяц назад", "Начало года", "Начало истории"])

### 1.3. Папки и проверка, что отчёт закрыт

Если `отчёт.xlsx` открыт в Excel, Python не сможет его перезаписать. Проверяем это сразу, а не в конце, после всей загрузки.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "архив").mkdir(parents=True, exist_ok=True)

if REPORT_FILE.exists():
    try:
        with open(REPORT_FILE, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"Закрой файл {REPORT_FILE} в Excel и перезапусти notebook")

print("Папки готовы, файл отчёта не заблокирован")

### 1.4. Журнал

Сюда записывается, что загрузилось и какие были проблемы. В конце журнал попадёт на лист «Лог» в отчёте.

In [ ]:
log = []

def add_to_log(name, status, message):
    log.append({"Показатель": name, "Статус": status, "Сообщение": message})

### 1.5. Вспомогательные функции

**Числа с сайта ЦБ.** ЦБ пишет числа как `3 732,7`: десятичная запятая, тысячи через пробел, причём иногда через неразрывный пробел, который pandas сам не распознаёт. Функция `to_number` приводит любой такой текст к числу. Если значение уже число, оно не портится; если в ячейке прочерк, получится пропуск (`NaN`).

**Архив.** Каждый показатель хранится в `data/<название>.csv`. Схема у всех загрузчиков одна:
1. прочитать архив (`read_archive`);
2. определить, с какой даты загружать (`download_start`): с последней даты архива минус 10 дней (чтобы подхватить исправления задним числом) или, если архива нет, с начала истории;
3. загрузить свежие данные;
4. склеить с архивом, свежие значения важнее (`update_archive`).

Если загрузка не удалась, используется то, что уже лежит в архиве, а в журнал пишется ошибка.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}    # сайт ЦБ иногда не отвечает «не браузерам»


def to_number(column):
    """'3 732,7' (с обычным или неразрывным пробелом) -> 3732.7"""
    text = column.astype(str).str.replace(r"[\s\xa0]", "", regex=True).str.replace(",", ".")
    return pd.to_numeric(text, errors="coerce")


def read_archive(name):
    """Архив показателя из data/<name>.csv (или пустая таблица, если архива ещё нет)"""
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        return pd.DataFrame(columns=["date"])
    return pd.read_csv(path, parse_dates=["date"])


def download_start(archive):
    """С какой даты загружать: с конца архива минус 10 дней или с начала истории"""
    # допуск 40 дней: у месячных рядов дата стоит на конце месяца, поэтому архив "начинается" позже начала истории
    archive_is_incomplete = archive.empty or archive["date"].min() > history_start + pd.Timedelta(days=40)
    if archive_is_incomplete:
        return history_start
    return min(archive["date"].max() - pd.Timedelta(days=10), report_date)


def update_archive(name, archive, fresh, key=["date"]):
    """Склеить архив со свежими данными (свежие важнее), сохранить и вернуть.
    key — по каким колонкам искать повторы: обычно дата, у КБД — дата и срок"""
    if archive.empty:
        combined = fresh
    elif fresh.empty:
        combined = archive
    else:
        combined = pd.concat([archive, fresh])

    combined["date"] = pd.to_datetime(combined["date"]).astype("datetime64[ns]")   # единый тип дат во всех таблицах
    combined = (combined
                .drop_duplicates(subset=key, keep="last")
                .sort_values(key)
                .reset_index(drop=True))
    combined.to_csv(DATA_DIR / f"{name}.csv", index=False)
    return combined

## 2. Загрузка данных
### 2.1. Функции загрузки: ЦБ

Три функции под три формата данных ЦБ:

| Функция | Что загружает | Формат |
|---|---|---|
| `load_cbr_table` | любая таблица из `cbr.ru/hd_base` вида «дата + значения»: ключевая ставка, RUONIA, ликвидность | HTML-таблица |
| `load_cbr_inflation` | инфляция г/г и цель | HTML-таблица, даты вида `09.2026` |
| `load_cbr_currency` | официальный курс валюты | XML |

Каждая возвращает таблицу с колонкой `date` и колонками значений.

In [ ]:
def load_cbr_table(page, value_column, name):
    """Таблица из cbr.ru/hd_base/<page>/: первый столбец — дата, value_column — нужное значение"""
    archive = read_archive(name)
    start = download_start(archive)

    params = {
        "UniDbQuery.Posted": "True",
        "UniDbQuery.From": start.strftime("%d.%m.%Y"),
        "UniDbQuery.To": report_date.strftime("%d.%m.%Y"),
    }
    try:
        resp = requests.get(f"https://www.cbr.ru/hd_base/{page}/", params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        table = pd.read_html(StringIO(resp.text), decimal=",", thousands=" ")[0]

        fresh = table.iloc[:, [0, value_column]].copy()
        fresh.columns = ["date", name]
        fresh["date"] = pd.to_datetime(fresh["date"], format="%d.%m.%Y")
        fresh[name] = to_number(fresh[name])
        add_to_log(name, "OK", f"загружено строк: {len(fresh)}")
    except Exception as error:
        fresh = pd.DataFrame(columns=["date", name])
        add_to_log(name, "ОШИБКА", f"не загрузилось, используется архив ({error})")

    time.sleep(0.5)
    return update_archive(name, archive, fresh)


def load_cbr_inflation():
    """Инфляция г/г и цель по инфляции. Месяц обозначается последним днём: 09.2026 -> 30.09.2026"""
    name = "inflation"
    archive = read_archive(name)
    start = download_start(archive)

    params = {
        "UniDbQuery.Posted": "True",
        "UniDbQuery.From": start.strftime("%d.%m.%Y"),
        "UniDbQuery.To": report_date.strftime("%d.%m.%Y"),
    }
    try:
        resp = requests.get("https://www.cbr.ru/hd_base/infl/", params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        # converters: дату читаем как текст, иначе "09.2026" превратится в число 9.2026
        table = pd.read_html(StringIO(resp.text), decimal=",", thousands=" ", converters={"Дата": str})[0]

        # колонки на сайте: Дата | Ключевая ставка | Инфляция | Цель (ключевую берём отдельно, дневную)
        fresh = table.iloc[:, [0, 2, 3]].copy()
        fresh.columns = ["date", "inflation", "inflation_target"]
        fresh["date"] = pd.to_datetime(fresh["date"], format="%m.%Y") + pd.offsets.MonthEnd(0)
        fresh["inflation"] = to_number(fresh["inflation"])
        fresh["inflation_target"] = to_number(fresh["inflation_target"])
        add_to_log(name, "OK", f"загружено строк: {len(fresh)}")
    except Exception as error:
        fresh = pd.DataFrame(columns=["date", "inflation", "inflation_target"])
        add_to_log(name, "ОШИБКА", f"не загрузилось, используется архив ({error})")

    time.sleep(0.5)
    return update_archive(name, archive, fresh)


def load_cbr_currency(code, name):
    """Официальный курс ЦБ за 1 единицу валюты. code — внутренний код ЦБ (R01235 — доллар)"""
    archive = read_archive(name)
    start = download_start(archive)

    params = {
        "date_req1": start.strftime("%d/%m/%Y"),
        "date_req2": report_date.strftime("%d/%m/%Y"),
        "VAL_NM_RQ": code,
    }
    try:
        resp = requests.get("https://www.cbr.ru/scripts/XML_dynamic.asp", params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)

        records = [
            {
                "date": record.get("Date"),
                name: float(record.find("Value").text.replace(",", ".")) / int(record.find("Nominal").text),
            }
            for record in root.findall("Record")
        ]
        fresh = pd.DataFrame(records, columns=["date", name])
        fresh["date"] = pd.to_datetime(fresh["date"], format="%d.%m.%Y")
        add_to_log(name, "OK", f"загружено строк: {len(fresh)}")
    except Exception as error:
        fresh = pd.DataFrame(columns=["date", name])
        add_to_log(name, "ОШИБКА", f"не загрузилось, используется архив ({error})")

    time.sleep(0.5)
    return update_archive(name, archive, fresh)

### 2.2. Загрузка: ЦБ

Здесь видно, что именно загружается. Номер столбца (`value_column`) — позиция нужного значения в таблице на сайте, считая с нуля: 0 — дата, 1 — следующий столбец.

Ликвидность — со знаком ЦБ: **плюс — дефицит, минус — профицит**, млрд руб.

In [ ]:
key_rate  = load_cbr_table("KeyRate",         value_column=1, name="key_rate")
ruonia    = load_cbr_table("ruonia/dynamics", value_column=1, name="ruonia")
liquidity = load_cbr_table("bliquidity",      value_column=1, name="liquidity")
inflation = load_cbr_inflation()
usdrub    = load_cbr_currency("R01235", name="usdrub")

### 2.3. Функции загрузки: Мосбиржа и ROISfix

| Функция | Что загружает | Особенность |
|---|---|---|
| `load_iss_index` | история индекса Мосбиржи: RUSFAR, корпоративные индексы | ответ порциями по 100 строк |
| `load_kbd_month_ends` | КБД на концы месяцев | один запрос на каждую дату; уже загруженные месяцы не перекачиваются |
| `load_roisfix` | ROISfix по всем срокам (1W–2Y) | CSV; строка заголовка и разделитель определяются автоматически |

In [ ]:
def load_iss_index(secid, columns, name):
    """История индекса Мосбиржи. columns — какие колонки ISS взять и как их назвать"""
    archive = read_archive(name)
    start = download_start(archive)
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{secid}.json"

    try:
        pages = []
        position = 0
        while True:
            params = {
                "from": start.strftime("%Y-%m-%d"),
                "till": report_date.strftime("%Y-%m-%d"),
                "start": position,
                "iss.meta": "off",
                "iss.only": "history",
            }
            resp = requests.get(url, params=params, timeout=60)
            resp.raise_for_status()
            block = resp.json()["history"]
            page = pd.DataFrame(block["data"], columns=block["columns"])
            if page.empty:
                break
            pages.append(page)
            position += len(page)
            time.sleep(0.3)

        if not pages:
            raise ValueError("биржа вернула пустой ответ")

        history = pd.concat(pages)
        fresh = history[["TRADEDATE"] + list(columns)].rename(columns={"TRADEDATE": "date", **columns})
        fresh["date"] = pd.to_datetime(fresh["date"])
        for column in columns.values():
            fresh[column] = pd.to_numeric(fresh[column], errors="coerce")   # пустые значения биржи -> NaN
        fresh = fresh.drop_duplicates(subset="date")
        add_to_log(name, "OK", f"загружено строк: {len(fresh)}")
    except Exception as error:
        fresh = pd.DataFrame(columns=["date"] + list(columns.values()))
        add_to_log(name, "ОШИБКА", f"не загрузилось, используется архив ({error})")

    return update_archive(name, archive, fresh)


def load_kbd_month_ends(month_ends):
    """КБД на каждую дату из month_ends. Если день неторговый, биржа отдаёт кривую
    на последний торговый день до него — его дата сохраняется в колонке tradedate"""
    name = "kbd"
    archive = read_archive(name)
    already_loaded = set(archive["date"]) if not archive.empty else set()

    # перекачиваем только недостающие месяцы; отчётную дату — всегда, на случай исправлений
    to_download = [date for date in month_ends if date not in already_loaded or date == report_date]

    curves = []
    for date in to_download:
        try:
            params = {"date": date.strftime("%Y-%m-%d"), "iss.meta": "off"}
            resp = requests.get("https://iss.moex.com/iss/engines/stock/zcyc.json", params=params, timeout=60)
            resp.raise_for_status()
            block = resp.json()["yearyields"]
            curve = pd.DataFrame(block["data"], columns=block["columns"])
            curves.append(pd.DataFrame({
                "date": date,
                "tradedate": curve["tradedate"],
                "period": curve["period"],
                "value": curve["value"],
            }))
        except Exception as error:
            add_to_log(name, "ОШИБКА", f"КБД на {date:%d.%m.%Y} не загрузилась ({error})")
        time.sleep(0.3)

    fresh = pd.concat(curves) if curves else pd.DataFrame(columns=["date", "tradedate", "period", "value"])
    add_to_log(name, "OK", f"загружено кривых: {len(curves)} из {len(to_download)}")
    kbd_all = update_archive(name, archive, fresh, key=["date", "period"])

    # срок и доходность — строго числами (иначе не сработает интерполяция при расчёте спредов)
    kbd_all["period"] = pd.to_numeric(kbd_all["period"], errors="coerce")
    kbd_all["value"] = pd.to_numeric(kbd_all["value"], errors="coerce")
    return kbd_all


ROISFIX_TERMS = ["1W", "2W", "1M", "2M", "3M", "6M", "1Y", "2Y"]

def load_roisfix():
    """ROISfix по всем срокам с roisfix.ru (архив отдаёт весь период одним файлом)"""
    name = "roisfix"
    archive = read_archive(name)
    start = download_start(archive)

    params = {
        "date_from": start.strftime("%d-%m-%Y"),
        "date_to": report_date.strftime("%d-%m-%Y"),
        "format": "csv",
    }
    try:
        resp = requests.get("https://roisfix.ru/archive", params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()

        # кодировка файла заранее неизвестна: пробуем UTF-8, иначе windows-1251
        try:
            text = resp.content.decode("utf-8-sig")
        except UnicodeDecodeError:
            text = resp.content.decode("cp1251")

        # перед таблицей есть строка с названием архива — ищем строку заголовка по словам "Дата ставки"
        lines = text.splitlines()
        header_rows = [number for number, line in enumerate(lines) if "Дата ставки" in line]
        if not header_rows:
            raise ValueError(f"в файле не найден заголовок 'Дата ставки'. Начало файла: {text[:200]!r}")
        header_row = header_rows[0]

        # разделитель определяем по строке заголовка: тот, которого в ней больше всего
        header_line = lines[header_row]
        separator = max(["\t", ";", ","], key=header_line.count)

        table = pd.read_csv(StringIO(text), sep=separator, skiprows=header_row, decimal=",")
        table.columns = table.columns.str.strip()

        fresh = table[["Дата ставки"] + ROISFIX_TERMS].copy()
        fresh.columns = ["date"] + [f"roisfix_{term.lower()}" for term in ROISFIX_TERMS]
        # в файле дата через дефис (16-10-2026), а Excel показывает её через точку — принимаем оба варианта
        fresh["date"] = pd.to_datetime(fresh["date"].astype(str).str.replace("-", "."), format="%d.%m.%Y")
        for column in fresh.columns[1:]:
            fresh[column] = to_number(fresh[column])
        add_to_log(name, "OK", f"загружено строк: {len(fresh)}")
    except Exception as error:
        fresh = pd.DataFrame(columns=["date"] + [f"roisfix_{term.lower()}" for term in ROISFIX_TERMS])
        add_to_log(name, "ОШИБКА", f"не загрузилось, используется архив ({error})")

    return update_archive(name, archive, fresh)

### 2.4. Загрузка: Мосбиржа и ROISfix

- **RUSFAR** — овернайт, значение индекса и есть ставка.
- **Корпоративные индексы** — берём доходность (`YIELD`, % годовых) и дюрацию (`DURATION`, в днях). Дюрация нужна только для расчёта спреда к ОФЗ сопоставимого срока, на слайды не выносится.
  - `RUCBTRAAANS` — группа рейтинга AAA;
  - `RUCBTRANS` — группа рейтинга A.
- **КБД** — на концы месяцев: окно графиков плюс текущий месяц. Начало года попадает в окно автоматически.
- **ROISfix** — дневные данные всех сроков, в отчёт пойдут значения на нужные даты.

In [ ]:
rusfar   = load_iss_index("RUSFAR",      {"CLOSE": "rusfar"},                                  name="rusfar")
corp_aaa = load_iss_index("RUCBTRAAANS", {"YIELD": "yield_aaa", "DURATION": "duration_aaa"},   name="corp_aaa")
corp_a   = load_iss_index("RUCBTRANS",   {"YIELD": "yield_a",   "DURATION": "duration_a"},     name="corp_a")

kbd_month_ends = pd.date_range(end=report_date, periods=DAILY_WINDOW_MONTHS + 1, freq="ME")
kbd = load_kbd_month_ends(kbd_month_ends)

roisfix = load_roisfix()

### 2.5. Ручные прогнозы ставки

Файл `прогнозы_ручные.xlsx` лежит рядом с notebook. Одна строка — одна публикация на один год:

| Источник | Дата публикации | Год | Ставка, нижняя граница | Ставка, верхняя граница | Комментарий |
|---|---|---|---|---|---|
| Прогноз ЦБ | 12.09.2026 | 2027 | … | … | средняя ключевая за год |
| Консенсус | 05.09.2026 | 2027 | … | … | для консенсуса обе границы одинаковые |

Новые публикации добавляются строками вниз, старые не удаляются. В расчётах возьмём последнюю публикацию не позже отчётной даты.

In [ ]:
if FORECASTS_FILE.exists():
    manual_forecasts = pd.read_excel(FORECASTS_FILE)
    manual_forecasts.columns = ["source", "published", "year", "rate_low", "rate_high", "comment"]
    manual_forecasts = manual_forecasts.dropna(subset=["source"])     # пустые строки не нужны
    manual_forecasts["year"] = manual_forecasts["year"].astype("Int64")   # год — целым числом, а не 2027.0
    manual_forecasts["published"] = pd.to_datetime(manual_forecasts["published"], dayfirst=True)
    add_to_log("manual_forecasts", "OK", f"строк в файле: {len(manual_forecasts)}")
else:
    manual_forecasts = pd.DataFrame(columns=["source", "published", "year", "rate_low", "rate_high", "comment"])
    add_to_log("manual_forecasts", "ОШИБКА", f"файл {FORECASTS_FILE} не найден, прогнозы ЦБ и консенсуса не попадут в отчёт")

manual_forecasts

### 2.6. Что загрузилось

Сводка по каждому показателю и журнал загрузки. Если в журнале есть «ОШИБКА», данные взяты из архива прошлых запусков; при самом первом запуске такой показатель будет пустым.

In [ ]:
loaded = {
    "Ключевая ставка": key_rate,
    "RUONIA": ruonia,
    "Ликвидность": liquidity,
    "Инфляция": inflation,
    "USD/RUB": usdrub,
    "RUSFAR": rusfar,
    "Корп. индекс AAA": corp_aaa,
    "Корп. индекс A": corp_a,
    "ROISfix": roisfix,
}

pd.DataFrame({
    name: {
        "Строк": len(table),
        "С": table["date"].min(),
        "По": table["date"].max(),
        "Последнее значение": table.iloc[-1, 1] if len(table) else None,
    }
    for name, table in loaded.items()
}).T

КБД — отдельной таблицей: строки — сроки, колонки — концы месяцев.

In [ ]:
kbd.pivot(index="period", columns="date", values="value").round(2)

In [ ]:
pd.DataFrame(log)

## 3. Проверки качества

Каждый ряд проверяется по трём признакам:
- **есть ли данные** до отчётной даты;
- **свежесть** — последнее значение не слишком далеко от отчётной даты (для дневных рядов — неделя, для инфляции — два месяца, она публикуется с лагом);
- **правдоподобие** — значения в разумном диапазоне. Это ловит ошибки разбора вроде той, когда дата `09.2026` превратилась в число.

Ни одна проверка не останавливает notebook: проблемы попадают в журнал и на лист «Лог» в отчёте.

In [ ]:
def check_series(label, table, column, max_lag_days, low, high):
    """Проверка одного ряда: наличие, свежесть, диапазон значений"""
    if column not in table.columns:
        return {"Показатель": label, "Последняя дата": None, "Результат": "НЕТ ДАННЫХ"}
    available = table[table["date"] <= report_date].dropna(subset=[column])
    if available.empty:
        return {"Показатель": label, "Последняя дата": None, "Результат": "НЕТ ДАННЫХ"}

    last_date = available["date"].max()
    lag_days = (report_date - last_date).days
    out_of_range = available[(available[column] < low) | (available[column] > high)]

    problems = []
    if lag_days > max_lag_days:
        problems.append(f"данные устарели: последние на {last_date:%d.%m.%Y}")
    if len(out_of_range) > 0:
        problems.append(f"значений вне диапазона {low}…{high}: {len(out_of_range)}")

    return {
        "Показатель": label,
        "Последняя дата": last_date,
        "Результат": "; ".join(problems) if problems else "OK",
    }


CHECKS = [
    # показатель,               таблица,                   колонка,         лаг, дней   мин.     макс.
    ("Ключевая ставка",         key_rate,                  "key_rate",          7,       0,      50),
    ("Инфляция г/г",            inflation,                 "inflation",        62,      -5,      50),
    ("USD/RUB",                 usdrub,                    "usdrub",            7,      30,     300),
    ("RUONIA",                  ruonia,                    "ruonia",            7,       0,      50),
    ("RUSFAR",                  rusfar,                    "rusfar",            7,       0,      50),
    ("Ликвидность",             liquidity,                 "liquidity",         7,  -20000,   20000),
    ("Доходность индекса AAA",  corp_aaa,                  "yield_aaa",         7,       0,      60),
    ("Дюрация индекса AAA, дн.", corp_aaa,                 "duration_aaa",      7,      30,    5000),
    ("Доходность индекса A",    corp_a,                    "yield_a",           7,       0,      60),
    ("Дюрация индекса A, дн.",  corp_a,                    "duration_a",        7,      30,    5000),
    ("КБД, 10 лет",             kbd[kbd["period"] == 10],  "value",             7,       0,      50),
    ("ROISfix 1Y",              roisfix,                   "roisfix_1y",        7,       0,      50),
]

checks = pd.DataFrame([check_series(*check) for check in CHECKS])

for _, row in checks[checks["Результат"] != "OK"].iterrows():
    add_to_log(row["Показатель"], "ПРЕДУПРЕЖДЕНИЕ", row["Результат"])

checks

## 4. Расчёты
### 4.1. Общие функции

- `value_on(таблица, колонка, дата)` — последнее доступное значение **не позже** даты и дата этого значения. Если 31 декабря — выходной, автоматически возьмётся 30-е или 29-е. Данные после отчётной даты не используются никогда, поэтому отчёт за прошлый месяц всегда пересоберётся таким же.
- `change_between(новое, старое, как)` — изменение в нужных единицах:
  - `"bp"` — в базисных пунктах (ставки, доходности);
  - `"pp"` — в процентных пунктах (инфляция, реальная ставка);
  - `"pct"` — в процентах (курс);
  - `"abs"` — простая разница в единицах показателя (спреды в б.п., ликвидность в млрд руб.).
- `in_window(таблица, начало)` — обрезка ряда под окно графика.

In [ ]:
def value_on(table, column, date):
    """Последнее доступное значение не позже даты. Возвращает (значение, дата значения)"""
    if column not in table.columns:          # показатель не загрузился совсем — в отчёте будет «н/д»
        return None, None
    available = table[table["date"] <= date].dropna(subset=[column])
    if available.empty:
        return None, None
    last_row = available.iloc[-1]
    return last_row[column], last_row["date"]


CHANGE_UNITS = {"bp": "б.п.", "pp": "п.п.", "pct": "%"}

def change_between(new, old, how):
    """Изменение от old к new: bp — в б.п., pp — в п.п., pct — в %, abs — простая разница"""
    if new is None or old is None or pd.isna(new) or pd.isna(old):
        return None
    if how == "bp":
        return (new - old) * 100
    if how == "pct":
        return (new / old - 1) * 100
    return new - old            # pp и abs


daily_start = report_date - pd.DateOffset(months=DAILY_WINDOW_MONTHS)
monthly_start = report_date - pd.DateOffset(months=MONTHLY_WINDOW_MONTHS)

def in_window(table, start):
    """Строки после start и не позже отчётной даты"""
    return table[(table["date"] > start) & (table["date"] <= report_date)].reset_index(drop=True)

### 4.2. Денежный рынок

Ключевая ставка, RUONIA и RUSFAR сводятся в одну таблицу по дате. Ключевая ставка действует каждый календарный день, поэтому на датах, где её нет в таблице ЦБ, протягиваем последнее значение. Спреды к ключевой — в б.п.

In [ ]:
money_market = (key_rate
                .merge(ruonia, on="date", how="outer")
                .merge(rusfar, on="date", how="outer")
                .sort_values("date")
                .reset_index(drop=True))

money_market["key_rate"] = money_market["key_rate"].ffill()
money_market["spread_ruonia_bp"] = (money_market["ruonia"] - money_market["key_rate"]) * 100
money_market["spread_rusfar_bp"] = (money_market["rusfar"] - money_market["key_rate"]) * 100

money_market.tail()

### 4.3. Макро: реальная ставка

К каждому месяцу инфляции подставляется ключевая ставка на конец этого месяца (`merge_asof` — «ближайшее значение не позже даты»). Реальная ставка = ключевая − инфляция г/г.

In [ ]:
macro_monthly = pd.merge_asof(inflation, key_rate, on="date")
macro_monthly["real_rate"] = macro_monthly["key_rate"] - macro_monthly["inflation"]

real_rate = macro_monthly[["date", "real_rate"]]

macro_monthly.tail()

### 4.4. КБД

- `kbd_table` — кривая на три даты по стандартным срокам и изменения в б.п.;
- `kbd_dates` — фактические торговые даты, на которые построены кривые (для подписи на слайде);
- `kbd_wide` — доходности по срокам в виде «одна строка — одна дата», нужна для сводки.

In [ ]:
KBD_TENORS = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20]

def kbd_curve(date):
    """КБД на конец месяца по стандартным срокам"""
    curve = kbd[kbd["date"] == date].set_index("period")["value"]
    return curve.reindex(KBD_TENORS)

kbd_now = kbd_curve(report_date)
kbd_month_ago = kbd_curve(month_ago)
kbd_year_start = kbd_curve(year_start)

kbd_table = pd.DataFrame({
    "Срок, лет": KBD_TENORS,
    "Отчётная дата": kbd_now.values,
    "Месяц назад": kbd_month_ago.values,
    "Начало года": kbd_year_start.values,
    "Изм. м/м, б.п.": ((kbd_now - kbd_month_ago) * 100).values,
    "Изм. с н.г., б.п.": ((kbd_now - kbd_year_start) * 100).values,
})

kbd_dates = pd.DataFrame({
    "Точка": ["Отчётная дата", "Месяц назад", "Начало года"],
    "Дата кривой": [kbd.loc[kbd["date"] == date, "tradedate"].iloc[0] if (kbd["date"] == date).any() else "н/д"
                    for date in [report_date, month_ago, year_start]],
})

kbd_wide = kbd.pivot(index="date", columns="period", values="value").reset_index()
kbd_wide.columns = ["date"] + [f"kbd_{period:g}y" for period in kbd_wide.columns[1:]]

display(kbd_dates)
kbd_table

### 4.5. Кредитные спреды

Спред = доходность корпоративного индекса − доходность КБД на сроке, равном дюрации индекса. Доходность КБД на нужном сроке — линейная интерполяция между соседними стандартными сроками. Считается на концы месяцев (там, где загружена КБД).

Проверка здравого смысла: дюрация в годах должна быть порядка 1–4 лет. Если в колонках `duration_..._years` окажутся сотые доли, значит биржа отдаёт дюрацию не в днях — скажи мне.

In [ ]:
def kbd_yield_at(date, maturity_years):
    """Доходность КБД на сроке maturity_years на дату (линейная интерполяция)"""
    curve = kbd[kbd["date"] == date].dropna(subset=["period", "value"]).sort_values("period")
    if curve.empty or maturity_years is None or pd.isna(maturity_years):
        return None
    return float(np.interp(float(maturity_years), curve["period"].astype(float), curve["value"].astype(float)))


def credit_spread_row(date):
    """Доходность, дюрация, КБД сопоставимого срока и спред для групп AAA и A на дату"""
    row = {"date": date}
    for group, table in [("aaa", corp_aaa), ("a", corp_a)]:
        index_yield, _ = value_on(table, f"yield_{group}", date)
        duration_days, _ = value_on(table, f"duration_{group}", date)
        duration_years = duration_days / 365 if duration_days is not None else None
        ofz_yield = kbd_yield_at(date, duration_years)

        row[f"yield_{group}"] = index_yield
        row[f"duration_{group}_years"] = duration_years
        row[f"kbd_{group}"] = ofz_yield
        row[f"spread_{group}_bp"] = change_between(index_yield, ofz_yield, "bp")
    return row

credit_spreads = pd.DataFrame([credit_spread_row(date) for date in kbd_month_ends])

credit_spreads.set_index("date").round(2)

### 4.6. Прогноз ключевой ставки

**Рыночная оценка по ROISfix.** ROISfix на срок — это ожидаемая рынком средняя RUONIA за этот срок. Добавив средний спред «ключевая − RUONIA» за последний месяц, получаем оценку средней ключевой ставки на срок. Оговорка для слайда: в ставках на длинные сроки есть премия за срок, поэтому оценка на 1–2 года немного завышает ожидания.

Считаем на отчётную дату и месяц назад, чтобы показать, как ожидания изменились.

**Консенсус и прогноз ЦБ** — последняя публикация каждого источника не позже отчётной даты.

In [ ]:
def market_rate_expectations(date):
    """Оценка средней ключевой ставки по срокам ROISfix на дату"""
    last_month = money_market[(money_market["date"] > date - pd.DateOffset(months=1)) & (money_market["date"] <= date)]
    key_minus_ruonia = (last_month["key_rate"] - last_month["ruonia"]).mean()

    roisfix_values = [value_on(roisfix, f"roisfix_{term.lower()}", date)[0] for term in ROISFIX_TERMS]
    return pd.Series(roisfix_values, index=ROISFIX_TERMS, dtype=float), key_minus_ruonia


roisfix_now, spread_now = market_rate_expectations(report_date)
roisfix_month_ago, spread_month_ago = market_rate_expectations(month_ago)

rate_expectations = pd.DataFrame({
    "Срок": ROISFIX_TERMS,
    "ROISfix, %": roisfix_now.values,
    "Оценка средней ключевой, %": (roisfix_now + spread_now).values,
    "Месяц назад, %": (roisfix_month_ago + spread_month_ago).values,
})
rate_expectations["Изм. за месяц, б.п."] = (
    rate_expectations["Оценка средней ключевой, %"] - rate_expectations["Месяц назад, %"]) * 100

print(f"Средний спред «ключевая − RUONIA» за последний месяц: {spread_now * 100:.0f} б.п.")
rate_expectations.round(2)

In [ ]:
published_by_report = manual_forecasts[manual_forecasts["published"] <= report_date]
latest_publication = published_by_report.groupby("source")["published"].transform("max")

latest_forecasts = (published_by_report[published_by_report["published"] == latest_publication]
                    .sort_values(["source", "year"])
                    [["source", "published", "year", "rate_low", "rate_high"]]
                    .rename(columns={
                        "source": "Источник",
                        "published": "Дата публикации",
                        "year": "Год",
                        "rate_low": "Ставка, нижняя граница",
                        "rate_high": "Ставка, верхняя граница",
                    }))

latest_forecasts

### 4.7. Сводка

Главная таблица отчёта. Каждая строка описана одной строкой в списке `SUMMARY`: чтобы добавить, убрать или переименовать показатель, достаточно поправить этот список. Порядок строк в списке = порядок строк в Excel.

Для месячных рядов (инфляция, реальная ставка) «месяц назад» считается от месяца последнего опубликованного значения. Если инфляция за сентябрь ещё не вышла, сравнение будет «август к июлю», а в колонке «Актуально на» будет стоять `08.2026`.

In [ ]:
MONTHLY_SERIES = {"inflation", "real_rate"}

def summary_row(section, label, unit, table, column, change):
    """Строка сводки: значения на три даты и изменения"""
    value, value_date = value_on(table, column, report_date)

    if column in MONTHLY_SERIES and value_date is not None:
        previous, _ = value_on(table, column, value_date - pd.offsets.MonthEnd(1))
        date_text = f"{value_date:%m.%Y}"
    else:
        previous, _ = value_on(table, column, month_ago)
        date_text = f"{value_date:%d.%m.%Y}" if value_date is not None else "н/д"

    at_year_start, _ = value_on(table, column, year_start)

    return {
        "Раздел": section,
        "Показатель": label,
        "Ед.": unit,
        "Актуально на": date_text,
        "Значение": value,
        "Месяц назад": previous,
        "Начало года": at_year_start,
        "Изм. м/м": change_between(value, previous, change),
        "Изм. с н.г.": change_between(value, at_year_start, change),
        "Ед. изменения": unit if change == "abs" else CHANGE_UNITS[change],
    }


MACRO, MONEY, BONDS = "Макроэкономика и ДКП", "Денежный рынок", "Облигационный рынок"

SUMMARY = [
    # раздел, показатель,                                  ед.,         таблица,        колонка,            изменение
    (MACRO,  "Ключевая ставка",                            "%",         key_rate,       "key_rate",         "bp"),
    (MACRO,  "Инфляция г/г",                               "%",         inflation,      "inflation",        "pp"),
    (MACRO,  "Реальная ключевая ставка",                   "%",         real_rate,      "real_rate",        "pp"),
    (MACRO,  "ROISfix 1Y (ожидания рынка)",                "%",         roisfix,        "roisfix_1y",       "bp"),
    (MACRO,  "Курс USD/RUB",                               "руб.",      usdrub,         "usdrub",           "pct"),
    (MONEY,  "RUONIA",                                     "%",         ruonia,         "ruonia",           "bp"),
    (MONEY,  "RUSFAR ON",                                  "%",         rusfar,         "rusfar",           "bp"),
    (MONEY,  "Спред RUONIA к ключевой",                    "б.п.",      money_market,   "spread_ruonia_bp", "abs"),
    (MONEY,  "Спред RUSFAR к ключевой",                    "б.п.",      money_market,   "spread_rusfar_bp", "abs"),
    (MONEY,  "Ликвидность: дефицит (+) / профицит (−)",    "млрд руб.", liquidity,      "liquidity",        "abs"),
    (BONDS,  "КБД 1 год",                                  "%",         kbd_wide,       "kbd_1y",           "bp"),
    (BONDS,  "КБД 3 года",                                 "%",         kbd_wide,       "kbd_3y",           "bp"),
    (BONDS,  "КБД 5 лет",                                  "%",         kbd_wide,       "kbd_5y",           "bp"),
    (BONDS,  "КБД 10 лет",                                 "%",         kbd_wide,       "kbd_10y",          "bp"),
    (BONDS,  "Доходность корп. облигаций AAA",             "%",         corp_aaa,       "yield_aaa",        "bp"),
    (BONDS,  "Доходность корп. облигаций A",               "%",         corp_a,         "yield_a",          "bp"),
    (BONDS,  "Спред AAA к ОФЗ",                            "б.п.",      credit_spreads, "spread_aaa_bp",    "abs"),
    (BONDS,  "Спред A к ОФЗ",                              "б.п.",      credit_spreads, "spread_a_bp",      "abs"),
]

summary = pd.DataFrame([summary_row(*row) for row in SUMMARY])

summary.round(2)

### 4.8. Ряды для графиков

Обрезаем ряды под окна графиков: дневные — за последние 12 месяцев, месячные — за 36. Кредитные спреды и КБД уже рассчитаны на 13 концов месяцев.

In [ ]:
chart_macro = in_window(macro_monthly, monthly_start)[["date", "key_rate", "inflation", "inflation_target", "real_rate"]]
chart_usdrub = in_window(usdrub, daily_start)
chart_money_market = in_window(money_market, daily_start)[["date", "key_rate", "ruonia", "rusfar", "spread_ruonia_bp", "spread_rusfar_bp"]]
chart_liquidity = in_window(liquidity, daily_start)
chart_credit = credit_spreads[["date", "yield_aaa", "kbd_aaa", "spread_aaa_bp", "yield_a", "kbd_a", "spread_a_bp"]]

pd.DataFrame({
    "Строк": [len(chart_macro), len(chart_usdrub), len(chart_money_market), len(chart_liquidity), len(chart_credit)],
}, index=["Макро (мес.)", "USD/RUB (дн.)", "Денежный рынок (дн.)", "Ликвидность (дн.)", "Кредитные спреды (мес.)"])

## 5. Запись в Excel

Раскладка отчёта фиксированная, чтобы ссылки из презентации не ломались: на каждом листе блоки стоят на одних и тех же местах — заголовок блока в строке 1, таблица со строки 3, блоки начинаются в колонках **A**, **M** и **Y**.

| Лист | Колонка A | Колонка M | Колонка Y |
|---|---|---|---|
| Сводка | сводная таблица | ключевые даты | — |
| Макро_ДКП | ключевая ставка, инфляция, реальная ставка (мес.) | USD/RUB (дн.) | — |
| Денежный_рынок | ключевая, RUONIA, RUSFAR, спреды (дн.) | ликвидность (дн.) | — |
| Облигации | КБД на три даты | даты кривых | доходности и кредитные спреды (мес.) |
| Прогноз_ставки | оценка рынка по ROISfix | консенсус и прогноз ЦБ | ключевая ставка, история (мес.) |
| Лог | проверки качества | журнал загрузки | — |

Названия колонок переводятся на русский по словарю `COLUMN_NAMES`.

In [ ]:
COLUMN_NAMES = {
    "date": "Дата",
    "key_rate": "Ключевая ставка, %",
    "inflation": "Инфляция г/г, %",
    "inflation_target": "Цель по инфляции, %",
    "real_rate": "Реальная ставка, %",
    "usdrub": "USD/RUB, руб.",
    "ruonia": "RUONIA, %",
    "rusfar": "RUSFAR ON, %",
    "spread_ruonia_bp": "Спред RUONIA к ключевой, б.п.",
    "spread_rusfar_bp": "Спред RUSFAR к ключевой, б.п.",
    "liquidity": "Ликвидность: дефицит (+) / профицит (−), млрд руб.",
    "yield_aaa": "Доходность AAA, %",
    "kbd_aaa": "КБД на дюрации AAA, %",
    "spread_aaa_bp": "Спред AAA к ОФЗ, б.п.",
    "yield_a": "Доходность A, %",
    "kbd_a": "КБД на дюрации A, %",
    "spread_a_bp": "Спред A к ОФЗ, б.п.",
}

BLOCK_COLUMNS = {"A": 0, "M": 12, "Y": 24}      # где начинаются блоки на листе (номер колонки с нуля)

def write_block(writer, sheet, table, title, position):
    """Блок на лист: заголовок в строке 1, таблица со строки 3, начиная с колонки position (A, M или Y)"""
    start_column = BLOCK_COLUMNS[position]
    rounded = table.copy()
    numeric_columns = rounded.select_dtypes("number").columns
    rounded[numeric_columns] = rounded[numeric_columns].round(4)
    rounded.rename(columns=COLUMN_NAMES).to_excel(
        writer, sheet_name=sheet, startrow=2, startcol=start_column, index=False)
    worksheet = writer.sheets[sheet]
    title_cell = worksheet.cell(row=1, column=start_column + 1, value=title)
    title_cell.font = Font(bold=True, size=12)

    # даты показываем как ДД.ММ.ГГГГ, без времени
    for offset, column in enumerate(rounded.columns):
        if pd.api.types.is_datetime64_any_dtype(rounded[column]):
            for row in range(4, 4 + len(rounded)):
                worksheet.cell(row=row, column=start_column + offset + 1).number_format = "DD.MM.YYYY"


key_dates_table = pd.DataFrame({
    "Точка": ["Отчётная дата", "Месяц назад", "Начало года"],
    "Дата": [report_date, month_ago, year_start],
})
key_rate_history = chart_macro[["date", "key_rate"]]

with pd.ExcelWriter(REPORT_FILE, engine="openpyxl", date_format="DD.MM.YYYY", datetime_format="DD.MM.YYYY") as writer:
    write_block(writer, "Сводка", summary, f"Обзор рынка за {report_period.strftime('%m.%Y')}", "A")
    write_block(writer, "Сводка", key_dates_table, "Ключевые даты", "M")

    write_block(writer, "Макро_ДКП", chart_macro, "Ключевая ставка, инфляция, реальная ставка (на конец месяца)", "A")
    write_block(writer, "Макро_ДКП", chart_usdrub, "Курс USD/RUB (официальный курс ЦБ)", "M")

    write_block(writer, "Денежный_рынок", chart_money_market, "Ключевая ставка, RUONIA, RUSFAR и спреды", "A")
    write_block(writer, "Денежный_рынок", chart_liquidity, "Ликвидность банковского сектора", "M")

    write_block(writer, "Облигации", kbd_table, "КБД ОФЗ на три даты", "A")
    write_block(writer, "Облигации", kbd_dates, "Даты кривых", "M")
    write_block(writer, "Облигации", chart_credit, "Корпоративные облигации: доходности и спреды к ОФЗ (на конец месяца)", "Y")

    write_block(writer, "Прогноз_ставки", rate_expectations, "Рыночная оценка средней ключевой ставки (ROISfix)", "A")
    write_block(writer, "Прогноз_ставки", latest_forecasts, "Консенсус и прогноз ЦБ (средняя ключевая за год)", "M")
    write_block(writer, "Прогноз_ставки", key_rate_history, "Ключевая ставка: история (на конец месяца)", "Y")

    write_block(writer, "Лог", checks, "Проверки качества", "A")
    write_block(writer, "Лог", pd.DataFrame(log), "Журнал загрузки", "M")

    # ширина колонок — по самому длинному значению (без заголовков блоков в строке 1), но не больше 50
    for worksheet in writer.sheets.values():
        for column_cells in worksheet.columns:
            lengths = [len(str(cell.value)) for cell in column_cells[1:] if cell.value is not None]
            worksheet.column_dimensions[column_cells[0].column_letter].width = min(max(lengths, default=8) + 2, 50)

print(f"Отчёт записан: {REPORT_FILE}")

### 5.1. Архивная копия и итог

Копия отчёта сохраняется в `output/архив/отчёт_<месяц>.xlsx`. Презентация всегда ссылается на `output/отчёт.xlsx`, поэтому ссылки не ломаются, а история отчётов сохраняется.

In [ ]:
archive_copy = OUTPUT_DIR / "архив" / f"отчёт_{REPORT_MONTH}.xlsx"
shutil.copy(REPORT_FILE, archive_copy)

problems = [entry for entry in log if entry["Статус"] != "OK"]

print(f"Отчётный месяц: {REPORT_MONTH}")
print(f"Отчёт:          {REPORT_FILE}")
print(f"Архивная копия: {archive_copy}")
print(f"Ошибок и предупреждений: {len(problems)}" + (" — подробности на листе «Лог»" if problems else ""))

pd.DataFrame(problems)

## 6. ИИ-комментарии (учебная часть)

GigaChat пишет тезисы по каждому разделу отчёта. Данные модель берёт сама — через инструменты MCP-сервера (`mcp_server.py`), который читает архивный отчёт за месяц. Затем код проверяет тезисы: каждое число должно быть в полученных данных, а состояние ликвидности (дефицит или профицит) — совпадать с данными. При расхождении модель исправляет текст. Результат — `output/тезисы_ГГГГ-ММ.xlsx`.

Рабочий отчёт от этого раздела не зависит: если GigaChat недоступен, `отчёт.xlsx` уже записан. Чтобы пропустить раздел, поставь `RUN_AI_COMMENTARY = False` в параметрах.

ИИ-часть запускается отдельным скриптом `ai_commentary.py`: MCP-клиент запускает сервер как отдельный процесс, а Jupyter на Windows этого не умеет.

**Подготовка (один раз):**
1. Установить библиотеки: `pip install "gigachat==0.2.3" "mcp==2.2.0"` (код проверен на этих версиях).
2. Положить рядом с notebook файлы `mcp_server.py` и `ai_commentary.py`.
3. Зарегистрироваться на developers.sber.ru → GigaChat API → создать проект; проверить, что начислен бесплатный пакет (Freemium).
4. Получить **ключ авторизации** и сохранить его одной строкой в файл `gigachat_key.txt` рядом с notebook. Это как пароль: никому не пересылать и не выкладывать вместе с проектом.
5. Скачать корневой сертификат Минцифры с **gosuslugi.ru/crt** и положить в папку `certs/` рядом с notebook. Сертификат используется только клиентом GigaChat в `ai_commentary.py`: в Windows и в общий набор сертификатов Python он не устанавливается.

In [30]:
import os
import subprocess
import sys

def run_ai_script(*arguments):
    """Запуск ai_commentary.py тем же Python, что и notebook; вывод скрипта — сюда"""
    result = subprocess.run(
        [sys.executable, "ai_commentary.py", *arguments],
        capture_output=True, text=True, encoding="utf-8",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ОШИБКА:\n" + result.stderr[-3000:])
    return result.returncode == 0

### 6.1. Проверка подключения к GigaChat

Короткий запрос к модели без данных отчёта. Если здесь ошибка — дело в ключе или сертификате, а не в отчёте.

In [31]:
if RUN_AI_COMMENTARY:
    run_ai_script("--check")

Ответ GigaChat: Да, подключение работает.
Модель: GigaChat-2-Pro:2.0.30.01 | токенов: 35



### 6.2. Тезисы по разделам

В выводе видно, какие функции модель вызывала в каждом разделе и прошли ли тезисы проверку. Тезисы с пометкой «ЕСТЬ ЗАМЕЧАНИЯ» нужно проверить вручную.

Проверка ловит неверные числа и перепутанный дефицит/профицит, но не любые смысловые ошибки — тезисы перед отправкой всё равно стоит прочитать.

In [32]:
theses_file = OUTPUT_DIR / f"тезисы_{REPORT_MONTH}.xlsx"

if RUN_AI_COMMENTARY and run_ai_script(REPORT_MONTH):
    display(pd.read_excel(theses_file, sheet_name="Тезисы"))

MCP-сервер запущен, инструментов: 7: list_reports, get_summary, get_kbd, get_credit_spreads, get_rate_expectations, get_series, get_data_quality

Макроэкономика и ДКП
    вызов: get_summary({"month": "2026-09", "section": "Макроэкономика и ДКП"})
    вызов: get_rate_expectations({"month": "2026-09"})
    проверка: OK

Денежный рынок
    вызов: get_summary({"month": "2026-09", "section": "Денежный рынок"})
    проверка: OK

Облигационный рынок
    вызов: get_kbd({"month": "2026-09"})
    вызов: get_credit_spreads({"month": "2026-09"})
    замечания: числа 600 нет в данных — прошу исправить (попытка 1)
    вызов: get_summary({"month": "2026-09", "section": "Облигационный рынок"})
    проверка: OK

Тезисы записаны: D:\Finance\IT\output\тезисы_2026-09.xlsx



,Раздел,Тезисы,Проверка,Замечания,Исправлений,Токенов
0,Макроэкономика и ДКП,"• Ключевая ставка осталась неизменной на уровне 14%, снизившись с начала года на 200 базисных пунктов.\n• Годовая инфляция увеличилась до 6.35%, продолжив рост с начала года (+0.76 п.п.), однако месячный прирост составил всего 0.02 п.п.\n• Реальная ключевая ставка снизилась до 7.65% под влиянием роста инфляции, уменьшившись с начала года на 2.76 п.п.\n• Ожидаемая рынком средняя ключевая ставка на горизонте до 1 года снижается, что указывает на ожидания дальнейшего умеренного смягчения денежно-кредитной политики.",OK,NaN,0,3375
1,Денежный рынок,"• RUONIA выросла на 36 б.п. за месяц и достигла 14.04%, продолжив сближение с ключевой ставкой Банка России.\n• Спред RUONIA к ключевой ставке сократился до 4 б.п., тогда как месяцем ранее составлял отрицательные 32 б.п.\n• Ликвидность банковского сектора характеризуется дефицитом в размере 3223.4 млрд рублей, увеличившись за месяц на 645.1 млрд рублей.\n• RUSFAR ON вырос на 39 б.п. до 14.07%, демонстрируя схожую динамику с RUONIA.",OK,NaN,0,2585
2,Облигационный рынок,"• Доходности корпоративных облигаций класса A резко выросли до 24,41%, увеличившись за месяц на 447 б.п. \n• Кредитный спред облигаций класса A достиг исторического максимума — 968,1 б.п., продолжив рост с начала года. \n• Спред облигаций класса AAA снизился до 52,6 б.п., демонстрируя улучшение восприятия рисков инвесторами. \n• Государственные облигации (КБД) показали заметное укрепление доходностей на длинных сроках: +83,9–92,7 б.п. за месяц.",OK,NaN,1,8269
